In [1]:
import os
os.chdir(os.path.dirname(os.getcwd()))

In [2]:
import polars as pl
import pandas as pd
import numpy as np
def dummy_npwarn_decorator_factory():
  def npwarn_decorator(x):
    return x
  return npwarn_decorator
np._no_nep50_warning = getattr(np, '_no_nep50_warning', dummy_npwarn_decorator_factory)
from statsmodels.tsa.stattools import acf, pacf
from scipy.stats import pearsonr
from utils.metrics import crps, quantile_loss
from tqdm import tqdm

In [ ]:
logs = pd.read_csv('logs/m5/clusters_detailed_scores_23Apr.csv')
logs

,test_index,actual,quant_0.005,quant_0.025,quant_0.165,quant_0.25,quant_0.5,quant_0.75,quant_0.835,quant_0.975,quant_0.995,cluster_id,mode,natural_grad,stabilization,n_rounds,dist
0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,32,softplus,False,NaN,23,NegativeBinomial
1,1,1,0.0,0.0,0.0,0.0,1.0,2.0,3.0,6.0,7.0,32,softplus,False,NaN,23,NegativeBinomial
2,2,2,0.0,0.0,0.0,0.0,0.0,1.0,1.0,3.0,4.0,32,softplus,False,NaN,23,NegativeBinomial
3,3,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,32,softplus,False,NaN,23,NegativeBinomial
4,4,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,3.0,4.0,32,softplus,False,NaN,23,NegativeBinomial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23595,295,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,4.0,90,exp,False,NaN,1645,NegativeBinomial
23596,296,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,3.0,4.0,90,exp,False,NaN,1645,NegativeBinomial
23597,297,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,90,exp,False,NaN,1645,NegativeBinomial
23598,298,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,90,exp,False,NaN,1645,NegativeBinomial


In [22]:
logs_NB = logs.loc[logs['dist'] == 'NegativeBinomial'].reset_index(drop=True)
logs_NB = logs_NB.loc[logs_NB['mode'] == 'softplus'].reset_index(drop=True)
# cluster numbers larget than 80
logs_NB = logs_NB.loc[logs_NB['cluster_id'] > 80].reset_index(drop=True)
logs_NB 

,test_index,actual,quant_0.005,quant_0.025,quant_0.165,quant_0.25,quant_0.5,quant_0.75,quant_0.835,quant_0.975,quant_0.995,cluster_id,mode,natural_grad,stabilization,n_rounds,dist
0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,98,softplus,False,NaN,1503,NegativeBinomial
1,1,0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,4.0,7.0,98,softplus,False,NaN,1503,NegativeBinomial
2,2,1,0.0,0.0,0.0,0.0,0.0,1.0,2.0,3.0,4.0,98,softplus,False,NaN,1503,NegativeBinomial
3,3,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,3.0,4.0,98,softplus,False,NaN,1503,NegativeBinomial
4,4,4,0.0,0.0,0.0,0.0,1.0,2.0,2.0,4.0,6.0,98,softplus,False,NaN,1503,NegativeBinomial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,225,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,86,softplus,False,NaN,1095,NegativeBinomial
416,226,3,0.0,0.0,2.0,2.0,4.0,6.0,7.0,12.0,16.0,86,softplus,False,NaN,1095,NegativeBinomial
417,227,10,0.0,0.0,2.0,3.0,4.0,7.0,8.0,14.0,19.0,86,softplus,False,NaN,1095,NegativeBinomial
418,228,1,0.0,0.0,0.0,0.0,1.0,2.0,2.0,4.0,5.0,86,softplus,False,NaN,1095,NegativeBinomial


In [23]:
# calculate average quant_0.5 quantile loss
quantile_loss(0.005, logs_NB['actual'], logs_NB['quant_0.005']).mean() # 0.0000000000000000

np.float64(0.007261904761904761)

In [24]:
quantile_loss(0.5, logs_NB['actual'], logs_NB['quant_0.5']).mean() # 0.0000000000000000

np.float64(0.4595238095238095)

In [25]:
quantile_loss(0.75, logs_NB['actual'], logs_NB['quant_0.75']).mean() # 0.0000000000000000

np.float64(0.40595238095238095)

In [ ]:
# plot 